# Evaluation Node: Metrics
Evaluate `evaluation_metrics` outputs against benchmark labels and rerun the node on selected papers.

In [1]:
import os
import sys
import json
import re
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / 'nodes').exists():
        break
    if ROOT.parent == ROOT:
        break
    ROOT = ROOT.parent
if not (ROOT / 'nodes').exists():
    raise RuntimeError(f"Could not resolve project root from cwd: {Path.cwd()}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.dsrp_state import DSRPState

PILOT = ROOT / 'Pilot_Evaluation'
BENCHMARK_FILE = PILOT / 'DATA_sample_10' / 'Data Science Research Process (DSRP) Framework.csv'
RESULTS_FOLDER = PILOT / 'pilot_study_results'
VECTOR_DB_PATH = PILOT / 'pilot_study_10_vectors'
COLLECTION_NAME = 'pilot_study_10'
EMBEDDING_MODEL = 'text-embedding-3-small'
NODE_KEY = 'evaluation_metrics'

DIMENSIONS = [
    'evaluation_strategy',
    'learning_type',
    'problem_type',
    'evaluation_metrics_present',
    'validation_procedure',
    'effect_size_reported',
    'assumption_checks_reported',
]

LIST_DIMENSIONS = {'learning_type', 'problem_type', 'validation_procedure'}

print(f'Project root: {ROOT}')
print(f'Benchmark file exists: {BENCHMARK_FILE.exists()}')
print(f'Results folder exists: {RESULTS_FOLDER.exists()}')

Project root: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow
Benchmark file exists: True
Results folder exists: True


In [2]:
def normalize_text(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None

def normalize_paper_id(value):
    text = normalize_text(value)
    return text.lower() if text else None

def split_multi(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, list):
        values = value
    else:
        text = str(value).strip()
        if not text:
            return []
        if text.startswith('[') and text.endswith(']'):
            text = text[1:-1].strip()
        values = re.split(r'[;,]', text)
    out = []
    for item in values:
        t = normalize_text(item)
        if t:
            out.append(t)
    return out

def canonical_yes_no(value):
    text = normalize_text(value)
    if not text:
        return 'No'
    return 'Yes' if text.lower() in {'yes', 'true', '1', 'y'} else 'No'

def canonical_effect(value):
    text = normalize_text(value)
    if not text:
        return 'Not reported'
    mapping = {
        'reported': 'Reported',
        'not reported': 'Not reported',
        'not applicable': 'Not applicable',
        'not_applicable': 'Not applicable',
    }
    return mapping.get(text.lower(), text)

def canonical_strategy(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        'statistical model diagnostics': 'Statistical Model Diagnostics',
        'machine learning evaluation': 'Machine Learning Evaluation',
        'mixed evaluation': 'Mixed Evaluation',
    }
    return mapping.get(text.lower(), text)

def canonical_learning(value):
    mapping = {
        'supervised': 'Supervised',
        'supervised learning': 'Supervised',
        'unsupervised': 'Unsupervised',
        'unsupervised learning': 'Unsupervised',
        'reinforcement': 'Reinforcement',
        'reinforcement learning': 'Reinforcement',
        'optimization-based learning': 'Optimization-Based Learning',
        'optimisation-based learning': 'Optimization-Based Learning',
        'not applicable': 'Not applicable',
        'not_applicable': 'Not applicable',
    }
    out = []
    for item in split_multi(value):
        canonical = mapping.get(item.lower(), item)
        if canonical not in out:
            out.append(canonical)
    return sorted(out)

def canonical_problem(value):
    mapping = {
        'classification': 'Classification',
        'regression': 'Regression',
        'clustering': 'Clustering',
        'dimensionality reduction': 'Dimensionality Reduction',
        'dimensionality_reduction': 'Dimensionality Reduction',
        'reward optimisation': 'Reward Optimisation',
        'reward optimization': 'Reward Optimisation',
        'optimisation': 'Optimisation',
        'optimization': 'Optimisation',
        'not applicable': 'Not applicable',
        'not_applicable': 'Not applicable',
    }
    out = []
    for item in split_multi(value):
        canonical = mapping.get(item.lower(), item)
        if canonical not in out:
            out.append(canonical)
    return sorted(out)

def canonical_validation(value):
    mapping = {
        'not reported': 'Not reported',
        'train/test split': 'train/test split',
        'cross-validation': 'cross-validation',
        'hold-out dataset': 'hold-out dataset',
        'temporal split': 'temporal split',
    }
    out = []
    for item in split_multi(value):
        canonical = mapping.get(item.lower(), item)
        if canonical not in out:
            out.append(canonical)
    if not out:
        out = ['Not reported']
    return sorted(out)

def normalize_dimension(dim, value):
    if dim == 'evaluation_strategy':
        return canonical_strategy(value)
    if dim == 'learning_type':
        return canonical_learning(value)
    if dim == 'problem_type':
        return canonical_problem(value)
    if dim == 'validation_procedure':
        return canonical_validation(value)
    if dim == 'evaluation_metrics_present':
        return canonical_yes_no(value)
    if dim in {'effect_size_reported', 'assumption_checks_reported'}:
        return canonical_effect(value)
    return value

def format_dimension_value(dim, value):
    if dim in LIST_DIMENSIONS:
        return '; '.join(value) if value else '[]'
    return str(value) if value is not None else '(missing)'

def list_jaccard(gt_list, pred_list):
    gt_set = set(gt_list or [])
    pred_set = set(pred_list or [])
    if not gt_set and not pred_set:
        return 1.0
    return len(gt_set & pred_set) / len(gt_set | pred_set)

def load_agent_results(results_folder):
    agent_data = {}
    for paper_dir in sorted(results_folder.glob('*/')):
        if not paper_dir.is_dir():
            continue
        results_file = paper_dir / 'aggregated' / 'results.json'
        if not results_file.exists():
            continue
        with open(results_file, 'r', encoding='utf-8') as f:
            parsed = json.load(f)
            pid = normalize_paper_id(parsed.get('paper_id', paper_dir.name))
            outputs = parsed.get('dsrp_outputs', {})
            if pid:
                agent_data[pid] = outputs
    return agent_data

In [3]:
if not BENCHMARK_FILE.exists():
    raise FileNotFoundError(f"Ground truth file not found: {BENCHMARK_FILE}")

csv_mtime = pd.to_datetime(BENCHMARK_FILE.stat().st_mtime, unit='s')
print(f"Reading fresh ground truth from: {BENCHMARK_FILE}")
print(f"CSV last modified: {csv_mtime}")

benchmark_df = pd.read_csv(BENCHMARK_FILE, low_memory=False)
benchmark_df.columns = benchmark_df.columns.str.strip()
required_cols = ['Paper ID', 'Gatekeeper'] + DIMENSIONS
missing_cols = [c for c in required_cols if c not in benchmark_df.columns]
if missing_cols:
    raise ValueError(f'Missing required benchmark columns: {missing_cols}')

benchmark_eval = benchmark_df[required_cols].copy()
benchmark_eval['Paper ID'] = benchmark_eval['Paper ID'].apply(normalize_paper_id)
benchmark_eval['Gatekeeper'] = benchmark_eval['Gatekeeper'].astype(str).str.strip().str.lower()
for dim in DIMENSIONS:
    benchmark_eval[dim] = benchmark_eval[dim].apply(lambda v: normalize_dimension(dim, v))
benchmark_eval = benchmark_eval[benchmark_eval['Gatekeeper'] == 'include'].copy()
benchmark_eval = benchmark_eval.dropna(subset=['Paper ID']).reset_index(drop=True)

agent_data = load_agent_results(RESULTS_FOLDER)

comparison_rows = []
missing_agent_results = []

for _, row in benchmark_eval.iterrows():
    pid = row['Paper ID']
    if pid not in agent_data:
        missing_agent_results.append(pid)
        continue

    node_output = agent_data[pid].get(NODE_KEY, {})
    record = {'Paper ID': pid}
    match_flags = []

    for dim in DIMENSIONS:
        gt_value = row[dim]
        pred_value = normalize_dimension(dim, node_output.get(dim))
        is_match = gt_value == pred_value
        match_flags.append(is_match)

        record[f'GT_{dim}'] = format_dimension_value(dim, gt_value)
        record[f'Agent_{dim}'] = format_dimension_value(dim, pred_value)
        record[f'Match_{dim}'] = 'MATCH' if is_match else 'MISMATCH'
        if dim in LIST_DIMENSIONS:
            record[f'Jaccard_{dim}'] = list_jaccard(gt_value, pred_value)

    record['Match_All'] = 'MATCH' if all(match_flags) else 'MISMATCH'
    comparison_rows.append(record)

comparison_df = pd.DataFrame(comparison_rows)
print(f'Included benchmark rows: {len(benchmark_eval)}')
print(f'Comparable rows: {len(comparison_df)}')
if missing_agent_results:
    print(f'Missing outputs for: {missing_agent_results}')

if len(comparison_df):
    show_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
    print(comparison_df[show_cols].to_string(index=False))

Reading fresh ground truth from: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-05 14:14:58.237343073
Included benchmark rows: 7
Comparable rows: 7
 Paper ID Match_evaluation_strategy Match_learning_type Match_problem_type Match_evaluation_metrics_present Match_validation_procedure Match_effect_size_reported Match_assumption_checks_reported Match_All
 2018 - 8                  MISMATCH               MATCH              MATCH                            MATCH                      MATCH                      MATCH                            MATCH  MISMATCH
2019 - 33                     MATCH               MATCH              MATCH                            MATCH                      MATCH                   MISMATCH                         MISMATCH  MISMATCH
 2020 - 8                     MATCH               MATCH           MISMATCH                            M

In [ ]:
if len(comparison_df) == 0:
    raise ValueError('No rows available for evaluation.')

metrics_rows = []
for dim in DIMENSIONS:
    y_true = comparison_df[f'GT_{dim}'].tolist()
    y_pred = comparison_df[f'Agent_{dim}'].tolist()

    acc = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

    row = {
        'dimension': dim,
        'samples': len(y_true),
        'accuracy': acc,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
    }
    if dim in LIST_DIMENSIONS:
        row['avg_jaccard'] = comparison_df[f'Jaccard_{dim}'].mean()
    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows)
print(metrics_df.to_string(index=False))

mismatch_df = comparison_df[comparison_df['Match_All'] == 'MISMATCH'].copy()
print(f'Overall exact match: {(comparison_df["Match_All"] == "MATCH").sum()}/{len(comparison_df)}')
print(f'Total mismatched papers: {len(mismatch_df)}')

## Iteration Utility
Run `modelling_node` as dependency, then rerun only `evaluation_metrics_foundational_node`.

In [3]:
import importlib
import nodes.modelling_node as modelling_module
import nodes.evaluation_nodes.evaluation_metrics_foundational as metrics_module

def _extract_metrics_reasoning(node_output):
    if not isinstance(node_output, dict):
        return '(no reasoning found)', ''
    reasoning = (
        node_output.get('validated_reasoning')
        or node_output.get('reasoning_explanation')
        or node_output.get('reasoning')
        or '(no reasoning found)'
    )
    audit = node_output.get('audit_commentary', '')
    return str(reasoning), str(audit or '')

def _extract_metrics_bibliography(node_output):
    if not isinstance(node_output, dict):
        return []
    bib = node_output.get('validated_bibliography')
    if isinstance(bib, list):
        return bib
    bib = node_output.get('bibliography')
    if isinstance(bib, list):
        return bib
    return []

def run_metrics_iteration(paper_ids, llm_model='gpt-4o-mini', run_modelling_dependency=True, verbose=True):
    importlib.reload(modelling_module)
    importlib.reload(metrics_module)

    modelling_node = modelling_module.modelling_node
    metrics_node = metrics_module.evaluation_metrics_foundational_node

    benchmark_lookup = benchmark_eval.set_index('Paper ID')
    rows = []
    raw_outputs = {}

    original_dir = Path.cwd().resolve()
    prev_env_model = os.getenv('DSRP_LLM_MODEL')

    os.chdir(ROOT)
    os.environ['DSRP_LLM_MODEL'] = llm_model

    try:
        for paper_id in paper_ids:
            pid = normalize_paper_id(paper_id)
            if not pid:
                continue

            if pid not in benchmark_lookup.index:
                rows.append({'Paper ID': pid, 'Model': llm_model, 'Error': 'Paper ID not found in benchmark include set'})
                continue

            state = DSRPState(
                paper_id=pid,
                dsrp_outputs={},
                collection_name=COLLECTION_NAME,
                persist_directory=str(VECTOR_DB_PATH),
                embedding_model=EMBEDDING_MODEL,
                gatekeeper={},
            )

            try:
                if run_modelling_dependency:
                    out = modelling_node(state)
                    state['dsrp_outputs'] = out.get('dsrp_outputs', state['dsrp_outputs'])

                out = metrics_node(state)
                state['dsrp_outputs'] = out.get('dsrp_outputs', state['dsrp_outputs'])

                node_output = state['dsrp_outputs'].get(NODE_KEY, {})
                raw_outputs[pid] = node_output

                record = {'Paper ID': pid, 'Model': llm_model}
                flags = []
                for dim in DIMENSIONS:
                    gt_value = benchmark_lookup.loc[pid, dim]
                    pred_value = normalize_dimension(dim, node_output.get(dim))
                    match = gt_value == pred_value
                    flags.append(match)
                    record[f'GT_{dim}'] = format_dimension_value(dim, gt_value)
                    record[f'New_Agent_{dim}'] = format_dimension_value(dim, pred_value)
                    record[f'Match_{dim}'] = 'MATCH' if match else 'MISMATCH'

                record['Match_All'] = 'MATCH' if all(flags) else 'MISMATCH'
                rows.append(record)

                if verbose:
                    summary = ' | '.join([f"{d}: {record[f'Match_{d}']}" for d in DIMENSIONS])
                    print(f"{pid}: {summary} | Match_All: {record['Match_All']}")

            except Exception as e:
                rows.append({'Paper ID': pid, 'Model': llm_model, 'Error': str(e)})
                if verbose:
                    print(f'{pid}: ERROR -> {e}')

    finally:
        if prev_env_model is None:
            os.environ.pop('DSRP_LLM_MODEL', None)
        else:
            os.environ['DSRP_LLM_MODEL'] = prev_env_model
        os.chdir(original_dir)

    return pd.DataFrame(rows), raw_outputs

def show_metrics_label_comparison(df):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        print('No results to display.')
        return
    cols = ['Paper ID', 'Model']
    for dim in DIMENSIONS:
        cols.extend([f'GT_{dim}', f'New_Agent_{dim}', f'Match_{dim}'])
    if 'Match_All' in df.columns:
        cols.append('Match_All')
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False))

def view_existing_metrics_reasoning(retest_df, retest_outputs, paper_id, show_bibliography=False):
    pid = normalize_paper_id(paper_id)
    rows = retest_df[retest_df['Paper ID'].apply(normalize_paper_id) == pid]
    if len(rows) == 0:
        raise ValueError(f'Paper ID not found in retest_df: {paper_id}')

    row = rows.iloc[-1].to_dict()
    node_output = retest_outputs.get(pid, {})

    print('\n' + '=' * 110)
    print(f"PAPER: {row.get('Paper ID', '(missing)')} | MODEL: {row.get('Model', '(missing)')}")
    print('=' * 110)
    for dim in DIMENSIONS:
        print(f"{dim}")
        print(f"  GT    : {row.get(f'GT_{dim}', '(missing)')}")
        print(f"  Agent : {row.get(f'New_Agent_{dim}', '(missing)')}")
        print(f"  Match : {row.get(f'Match_{dim}', '(missing)')}")
    print(f"\nMatch_All: {row.get('Match_All', '(missing)')}")

    reasoning, audit = _extract_metrics_reasoning(node_output)
    print('\nReasoning:')
    print(reasoning)
    if audit.strip():
        print('\nAudit Commentary:')
        print(audit)

    if show_bibliography:
        bibliography = _extract_metrics_bibliography(node_output)
        print('\nBibliography:')
        if not bibliography:
            print('  (none)')
        else:
            for i, item in enumerate(bibliography, start=1):
                page = item.get('page', '') if isinstance(item, dict) else ''
                section = item.get('section', '') if isinstance(item, dict) else ''
                quote = item.get('direct_quote', '') if isinstance(item, dict) else ''
                print(f"  [{i}] page={page} | section={section}")
                print(f"      quote: {quote}")

def rerun_single_metrics_paper(paper_id, llm_model='gpt-4o-mini', run_modelling_dependency=True, verbose=True):
    df, outputs = run_metrics_iteration(
        [paper_id],
        llm_model=llm_model,
        run_modelling_dependency=run_modelling_dependency,
        verbose=verbose,
    )
    show_metrics_label_comparison(df)
    return df, outputs

inspect_metrics_reasoning = view_existing_metrics_reasoning

In [ ]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_metrics_iteration(
    paper_ids=all_included_papers,
    llm_model='gpt-4o-mini',
    run_modelling_dependency=True,
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_metrics_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

In [ ]:
# Helper usage examples

# A) View reasoning from existing retest results (NO rerun)
view_existing_metrics_reasoning(
    retest_df=retest_df,
    retest_outputs=retest_outputs,
    paper_id='2021 - 56',
    show_bibliography=False,
 )

# B) Re-run on a single paper only (optional)
# single_retest_df, single_retest_outputs = rerun_single_metrics_paper(
#     paper_id='2021 - 56',
#     llm_model='gpt-4o-mini',
#     run_modelling_dependency=True,
#     verbose=True,
# )